# Chapter 5: Real EEG, quality checks and spectral interpretation

Load the PhysioNet baseline recording, inspect 20 seconds, and calculate a Welch spectrum without making diagnostic claims.

**Education only, not clinical software.** Run cells in order and explain results and limitations.

In [ ]:
from pathlib import Path
import mne, numpy as np, matplotlib.pyplot as plt
from scipy.signal import welch
edf=Path("S001R01.edf")
assert edf.is_file(), "Download S001R01.edf from PhysioNet first; do not commit raw recordings."
raw=mne.io.read_raw_edf(str(edf),preload=False)
sf=float(raw.info["sfreq"]); picks=mne.pick_types(raw.info,eeg=True)[:5]
data,t=raw[picks,:min(raw.n_times,int(20*sf))]
fig,ax=plt.subplots(2,1,figsize=(12,7))
for i,signal in enumerate(data):
    ax[0].plot(t,signal*1e6+100*i,label=raw.ch_names[picks[i]])
    freq,psd=welch(signal,fs=sf,nperseg=min(len(signal),int(2*sf)))
    mask=(freq>=0.5)&(freq<=min(45,sf/2))
    ax[1].plot(freq[mask],10*np.log10(np.maximum(psd[mask]*1e12,1e-20)),label=raw.ch_names[picks[i]])
ax[0].set(xlabel="Time (s)",ylabel="µV + channel offsets",title="Raw excerpt")
ax[1].set(xlabel="Frequency (Hz)",ylabel="PSD (dB re 1 µV²/Hz)",title="Welch PSD")
for a in ax:a.legend()
plt.tight_layout();plt.show()
print("Review artifacts using EOG/ECG and acquisition context where available; no diagnosis.")

## Interpretation and reproducibility

1. What scientific quantity was measured or simulated, and in which units?
2. Which parameters, acquisition conditions, or assumptions affect the output?
3. What can you **not** conclude from this experiment?
4. Record versions, inputs, and observations; explain one reproducible extension.

**Graduate extension:** Change one justified parameter, compare the output quantitatively, and explain the difference without making unwarranted clinical claims.